# 02 — Data Cleaning & Reconciliation (Phase 1)

This notebook documents every transformation applied to the 6 non-geometry raw
datasets (poverty, IPM, population, education, expenditure, stunting) on the way to
one merged province x year analytical table. Boundaries (GADM) are excluded from this
merge — geospatial joins are Phase 4 — but the province-vintage gap is reconciled here
so Phase 4 doesn't have to re-discover it.

Every value shown below comes from real ingested data (`data/raw/`, see
`docs/data_inventory.md` for provenance). Nothing in this notebook is simulated.


In [1]:
import os
import sys

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import pandas as pd
from src.cleaning.standardize import standardize_dataset
from src.cleaning.profile_missing import profile_dataset, DATASET_VALUE_COLUMNS
from src.cleaning.merge import build_merged_table, validate_merged_table

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)


## Step 1 — Province Reconciliation

The canonical province set (38 provinces) is `src/reference/province_lookup.csv`,
generated from a live BPS statistical variable's region list (Phase 0B), not BPS's
stale `/domain/type/prov/` metadata endpoint. See `docs/province_reconciliation.md`
for the full methodology, including the GADM-boundary crosswalk for the 4 provinces
created by Indonesia's 2022 Papua split.


In [2]:
lookup = pd.read_csv("src/reference/province_lookup.csv")
print(f"{lookup['canonical_name'].nunique()} canonical provinces")

crosswalk = pd.read_csv("src/reference/province_gadm_crosswalk.csv")
crosswalk


38 canonical provinces


,province,gadm_parent_name,legal_basis
0,Papua Selatan,Papua,UU No. 14/2022
1,Papua Tengah,Papua,UU No. 15/2022
2,Papua Pegunungan,Papua,UU No. 16/2022
3,Papua Barat Daya,Papua Barat,UU No. 29/2022


## Step 2 — Cleaning: standardize each raw dataset

For each dataset: drop non-province aggregate rows (e.g. an `INDONESIA` national
total row that BPS includes alongside the 38 real provinces), map every province name
to its canonical form, and flag (never silently fix) duplicate or inconsistent
`(province, year)` records.


In [3]:
reports = {}
for slug, cols in DATASET_VALUE_COLUMNS.items():
    df, report = standardize_dataset(slug, cols)
    reports[slug] = report
    print(f"{slug:12s} rows_in={report.rows_in:3d}  dropped_non_province={report.rows_dropped_non_province}  "
          f"dropped_exact_dup={report.rows_dropped_exact_duplicate}  inconsistent_keys={len(report.inconsistent_keys)}  "
          f"rows_out={report.rows_out}")


poverty      rows_in= 78  dropped_non_province=2  dropped_exact_dup=0  inconsistent_keys=0  rows_out=76
ipm          rows_in=117  dropped_non_province=3  dropped_exact_dup=0  inconsistent_keys=0  rows_out=114
population   rows_in=105  dropped_non_province=3  dropped_exact_dup=0  inconsistent_keys=0  rows_out=102
education    rows_in=105  dropped_non_province=3  dropped_exact_dup=0  inconsistent_keys=0  rows_out=102
expenditure  rows_in=110  dropped_non_province=0  dropped_exact_dup=0  inconsistent_keys=0  rows_out=110
stunting     rows_in= 38  dropped_non_province=0  dropped_exact_dup=0  inconsistent_keys=0  rows_out=38


All datasets show zero inconsistent keys and at most a handful of dropped
`INDONESIA` aggregate rows — no genuine data-quality problems found, only the expected
aggregate-row removal.


## Step 3 — Missing-value assessment

Profile each cleaned dataset against the canonical 38-province set. A missing
province here means BPS has not published that particular series for it — it is never
imputed.


In [4]:
profiles = [profile_dataset(slug, cols) for slug, cols in DATASET_VALUE_COLUMNS.items()]
summary = pd.DataFrame(profiles)[["dataset", "n_provinces_present", "n_provinces_missing", "missing_provinces", "years_present"]]
summary


,dataset,n_provinces_present,n_provinces_missing,missing_provinces,years_present
0,poverty,38,0,[],"[2024, 2025]"
1,ipm,38,0,[],"[2022, 2023, 2024]"
2,population,34,4,"[Papua Barat Daya, Papua Pegunungan, Papua Sel...","[2018, 2019, 2020]"
3,education,34,4,"[Papua Barat Daya, Papua Pegunungan, Papua Sel...","[2021, 2022, 2023]"
4,expenditure,38,0,[],"[2023, 2024, 2025]"
5,stunting,38,0,[],[2024]


`population` and `education` are each missing exactly the 4 provinces created by the
2022 Papua split (Papua Selatan, Papua Tengah, Papua Pegunungan, Papua Barat Daya) —
BPS has not yet republished these two series broken out for them. Full detail and the
recommended per-dataset policy (always `leave_null`, never impute) is in
`docs/missing_data_report.md`.


## Step 4 — Merged analytical table

One row per province, with each indicator's most recent real value and that value's
own year column (datasets span different year ranges — see
`src/cleaning/merge.py`'s docstring for why a single shared "year" column would
require either dropping data or fabricating it).


In [5]:
merged = build_merged_table()
validate_merged_table(merged)
print(f"{merged.shape[0]} rows, {merged.shape[1]} columns — validation passed")
merged.head(10)


38 rows, 14 columns — validation passed


,province,poverty_rate,poverty_rate_year,ipm,ipm_year,population,population_year,participation_rate,participation_rate_year,expenditure_per_capita,expenditure_per_capita_year,stunting_rate,stunting_category,stunting_rate_year
0,Aceh,12.22,2025,74.03,2024,5388.1,2020.0,99.43,2023.0,11191,2025,28.6,(20%-29.9%) Medium,2024
1,Bali,3.42,2025,77.76,2024,4414.4,2020.0,99.61,2023.0,15383,2025,8.7,(0%-19.9%) Rendah,2024
2,Banten,5.51,2025,74.48,2024,12895.3,2020.0,99.43,2023.0,13498,2025,21.1,(20%-29.9%) Medium,2024
3,Bengkulu,11.88,2025,73.39,2024,1994.3,2020.0,99.42,2023.0,12197,2025,18.8,(0%-19.9%) Rendah,2024
4,Di Yogyakarta,10.08,2025,81.55,2024,3919.2,2020.0,99.63,2023.0,15855,2025,17.4,(0%-19.9%) Rendah,2024
5,Dki Jakarta,4.03,2025,83.08,2024,10576.4,2020.0,99.49,2023.0,20676,2025,17.3,(0%-19.9%) Rendah,2024
6,Gorontalo,12.62,2025,71.23,2024,1186.3,2020.0,98.69,2023.0,11937,2025,23.8,(20%-29.9%) Medium,2024
7,Jambi,6.89,2025,73.43,2024,3604.2,2020.0,99.49,2023.0,12018,2025,17.1,(0%-19.9%) Rendah,2024
8,Jawa Barat,6.78,2025,74.43,2024,49565.2,2020.0,99.40,2023.0,12447,2025,15.9,(0%-19.9%) Rendah,2024
9,Jawa Tengah,9.39,2025,73.88,2024,34738.2,2020.0,99.57,2023.0,12746,2025,17.1,(0%-19.9%) Rendah,2024


In [6]:
null_counts = merged.isna().sum()
null_counts[null_counts > 0]


population                 4
population_year            4
participation_rate         4
participation_rate_year    4
dtype: int64

In [7]:
merged[merged["province"].str.contains("Papua")][["province", "population", "participation_rate", "poverty_rate", "stunting_rate"]]


,province,population,participation_rate,poverty_rate,stunting_rate
23,Papua,3393.1,83.61,17.82,24.7
24,Papua Barat,986.0,98.41,19.58,24.6
25,Papua Barat Daya,NaN,NaN,17.50,30.5
26,Papua Pegunungan,NaN,NaN,27.21,40.0
27,Papua Selatan,NaN,NaN,19.26,25.8
28,Papua Tengah,NaN,NaN,29.45,32.5


The 4 new Papua provinces are null for exactly `population` and `participation_rate`
(and their year columns) — matching the missing-data assessment above — and populated
for every other indicator. This table is written to
`data/processed/merged_provincial_indicators.csv` by `python -m src.cleaning.merge`
(or `make` once a Phase 1 target is added).

## Summary

- 38 canonical provinces, reconciled against 3 distinct gap types (aggregate rows,
  genuine BPS publication gaps, GADM geometry vintage) — see
  `docs/province_reconciliation.md`
- 0 inconsistent or silently-dropped real province records across all 6 datasets
- 1 merged table, 38 rows, every value traceable to a real source and year
- No scoring, ranking, modeling, or dashboard work performed — out of scope for this
  phase
